<a href="https://colab.research.google.com/github/xxm-cmyk/SaudiCord/blob/main/example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!git clone https://github.com/philshiu/Drosophila_brain_model.git
%cd Drosophila_brain_model

!find . -type f \( -name "2023_03_23_completeness_630_final.csv" -o -name "2023_03_23_connectivity_630_final.parquet" \)

Cloning into 'Drosophila_brain_model'...
remote: Enumerating objects: 183, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 183 (delta 72), reused 69 (delta 69), pack-reused 105 (from 1)
Receiving objects: 100% (183/183), 185.06 MiB | 38.83 MiB/s, done.
Resolving deltas: 100% (82/82), done.
/content/Drosophila_brain_model
./2023_03_23_connectivity_630_final.parquet
./2023_03_23_completeness_630_final.csv


In [12]:
from pathlib import Path

Path("flymind_live.py").write_text('''from brian2 import Network, PoissonInput, SpikeMonitor, Hz
from model import create_model, default_params

class FlyMindLive:
    def __init__(self, path_comp, path_con, input_neurons):
        self.params = default_params.copy()

        self.neu, self.syn, self.monitor = create_model(
            path_comp, path_con, self.params
        )

        self.input_indices = list(input_neurons)

        self.inputs = [
            PoissonInput(
                target=self.neu[index],
                target_var="v",
                N=1,
                rate=0 * Hz,
                weight=self.params["w_syn"] * self.params["f_poi"]
            )
            for index in self.input_indices
        ]

        self.network = Network(
            self.neu,
            self.syn,
            self.monitor,
            *self.inputs
        )

        self.network.store("initial")

    def set_input_rate(self, rate_hz):
        for source in self.inputs:
            source.rate = rate_hz * Hz

    def step(self, duration_ms=50):
        self.network.run(duration_ms * 0.001)

        return {
            int(index): int(count)
            for index, count in zip(
                self.monitor.i[:],
                self.monitor.count[:]
            )
        }
''')

print("FlyMind live module created:", Path("flymind_live.py").exists())

FlyMind live module created: True


In [22]:
import importlib
import flymind_live

importlib.reload(flymind_live)

from flymind_live import FlyMindLive

live = FlyMindLive(
    comp_path,
    con_path,
    [flyid2i[n] for n in sugar_ids]
)

print("Live network ready")
print("Input neurons:", len(live.inputs) if hasattr(live, "inputs") else len(sugar_ids))

WARNING    The object 'default_neurons' is getting deleted, but was never included in a network. This probably means that you did not store the object reference in a variable, or that the variable was not used to construct the network.
The object was created here (most recent call only):
  File '/content/Drosophila_brain_model/model.py', line 160, in create_model
    neu = NeuronGroup( # create neurons [brian2.core.base.unused_brian_object]


Live network ready
Input neurons: 21


In [24]:
live.set_input_rate(75)

activity = live.step(50)

print("Active neurons:", len(activity))
print("Total new spikes:", sum(activity.values()))
print("MN9 new spikes:", activity.get(flyid2i[mn9_id], 0))

Active neurons: 76
Total new spikes: 159
MN9 new spikes: 0


In [31]:
mn9_grouped = (
    mn9_inputs.groupby("Presynaptic_ID")
    .agg(
        Connections=("Connectivity", "sum"),
        Excitatory=("Excitatory", "first")
    )
)

mn9_grouped["Spikes"] = [
    test_150.get(flyid2i[int(neuron_id)], 0)
    for neuron_id in mn9_grouped.index
]

active_direct = mn9_grouped[
    mn9_grouped["Spikes"] > 0
].sort_values("Spikes", ascending=False)

print("Unique input neurons:", len(mn9_grouped))
print("Active input neurons:", len(active_direct))
print("\nTop active inputs to MN9:")
print(active_direct.head(20).to_string())

Unique input neurons: 229
Active input neurons: 48

Top active inputs to MN9:
                    Connections  Excitatory  Spikes
Presynaptic_ID                                     
720575940615041430            1          -1      21
720575940615671106            2           1      16
720575940631918649            4          -1      15
720575940619877396           32           1      14
720575940623211725          424           1      12
720575940632252743          281           1      12
720575940607272649            3           1      11
720575940636053678            4          -1      11
720575940631349335            1           1      10
720575940616466260            1          -1      10
720575940649829241          173          -1       9
720575940625867056            1           1       9
720575940610677828           82          -1       9
720575940628826128            1          -1       9
720575940609407939           88           1       8
720575940616857174           25       

In [39]:
# FlyMind: خريطة مدخلات الحواس

SENSOR_INPUTS = {
    "vision": {
        "Center": 0,
        "Left": 0,
        "Right": 0,
        "FarLeft": 0,
        "FarRight": 0,
        "Up": 0,
        "Down": 0,
        "UpLeft": 0,
        "UpRight": 0
    },
    "body": {
        "Grounded": 0,
        "ForwardTilt": 0,
        "SideTilt": 0,
        "BalanceMagnitude": 0
    }
}

print("Vision sensors:", len(SENSOR_INPUTS["vision"]))
print("Body sensors:", len(SENSOR_INPUTS["body"]))
print("Total sensor signals:", sum(map(len, SENSOR_INPUTS.values())))

Vision sensors: 9
Body sensors: 4
Total sensor signals: 13


In [47]:
import importlib
import flymind_live

importlib.reload(flymind_live)

from flymind_live import FlyMindLive

print(hasattr(FlyMindLive, "set_body_signal"))

True


# Introduction
## Underlying connectivity data
The connectivity of the fly brain is stored in the folowing files:
- neurons present: `config['path_comp']`
- connectivity between neurons: `config['path_con]`

## Model parameters
The equation and constants for the leaky integrate and fire model are defined
in the dictionary `default_params` in the beginning of the file `model.py`:

```
default_params = {
    # trials
    't_run'     : 1000 * ms,              # duration of trial
    'n_run'     : 30,                     # number of runs

    'v_0'       : -52 * mV,               # resting potential
    'v_rst'     : -52 * mV,               # reset potential after spike
    [...]
```
We can also change values
and pass the modified dictionary to the model (see Experiment 1).

## Addressing neurons
Here, we want to stimulate some sugar-sensing neurons in the right hemisphere.
The neurons of interest are defined via their flywire IDs:

In [50]:
# الخلايا الهابطة المرشحة
output_neurons = {
    "DN1": 720575940616185531,
    "DN2": 720575940629806974
}

output_indices = {
    name: flyid2i[neuron_id]
    for name, neuron_id in output_neurons.items()
}

# تشغيل المحاكاة
live = FlyMindLive(comp_path, con_path, jon_indices)

live.set_body_signal(0.5)
activity = live.step(200)

# استخراج إشارات المخرج
outputs = {
    name: activity.get(index, 0)
    for name, index in output_indices.items()
}

print("Brain outputs:", outputs)

Brain outputs: {'DN1': 0, 'DN2': 0}


In [59]:
from google.colab import userdata
import requests

token = userdata.get("FLYMIND_TOKEN").strip()

response = requests.post(
    "https://flymind-api.ztrx1j.workers.dev/movement",
    headers={"Authorization": "Bearer " + token},
    json={
        "forward": movement["Forward"],
        "turn": movement["Turn"]
    },
    timeout=10
)

print(response.status_code)
print(response.json())

202
{'ok': True, 'saved': True, 'movement': {'forward': 0.558, 'turn': 0, 'updated_at': '2026-11-13T10:17:25.945Z'}}


In [45]:
groups = {
    "Sugar": neu_sugar,
    "Water": neu_water,
    "Bitter": neu_bitter,
    "JON": neu_JON_all
}

for name, ids in groups.items():
    found = sum(n in flyid2i for n in ids)
    print(f"{name}: {found}/{len(ids)}")

Sugar: 21/21
Water: 18/18
Bitter: 21/21
JON: 146/146


In [3]:
neu_sugar = [
    720575940624963786,
    720575940630233916,
    720575940637568838,
    720575940638202345,
    720575940617000768,
    720575940630797113,
    720575940632889389,
    720575940621754367,
    720575940621502051,
    720575940640649691,
    720575940639332736,
    720575940616885538,
    720575940639198653,
    720575940620900446,
    720575940617937543,
    720575940632425919,
    720575940633143833,
    720575940612670570,
    720575940628853239,
    720575940629176663,
    720575940611875570,
]

For an easier identification, we define also a mapping from the flywire IDs to custom
names. The above neurons are calles `sugar_1`, `sugar_2` etc:

In [4]:
flyid2name = { f: f'sugar_{i+1}' for i, f in enumerate(neu_sugar) }
flyid2name

{720575940624963786: 'sugar_1',
 720575940630233916: 'sugar_2',
 720575940637568838: 'sugar_3',
 720575940638202345: 'sugar_4',
 720575940617000768: 'sugar_5',
 720575940630797113: 'sugar_6',
 720575940632889389: 'sugar_7',
 720575940621754367: 'sugar_8',
 720575940621502051: 'sugar_9',
 720575940640649691: 'sugar_10',
 720575940639332736: 'sugar_11',
 720575940616885538: 'sugar_12',
 720575940639198653: 'sugar_13',
 720575940620900446: 'sugar_14',
 720575940617937543: 'sugar_15',
 720575940632425919: 'sugar_16',
 720575940633143833: 'sugar_17',
 720575940612670570: 'sugar_18',
 720575940628853239: 'sugar_19',
 720575940629176663: 'sugar_20',
 720575940611875570: 'sugar_21'}

# Running simulations
## Activating a set of neurons
To run a simulation exciting these nerons we have to call `run_exp` supplying the following:
- unique name for the simulation: `exp_name`
- a list of neurons we want to stimulate: `neu_sugar`
- the connectivity data: `config['path_comp']` and `config['path_con]`
- path to store the output: `config['path_res']`
- number of CPU cores use: `config['n_procs]`

Note that running this on Google Colab can take roughly 20 minutes; it is substantially faster on a local install, depending on the number of CPU cores. By default, the neurons are excited at 200 Hz.

In [5]:
# activate sugar sensing neurons
run_exp(exp_name='sugarR', neu_exc=neu_sugar, **config)

>>> Skipping experiment sugarR because results/example/sugarR.parquet exists and force_overwrite = False


In [42]:
from google.colab import files
files.download("flymind_motor_output.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [45]:
import requests
import getpass
import json

API = "https://flymind-api.ztrx1j.workers.dev"

token = getpass.getpass("FlyMind API Token: ")

with open("flymind_motor_output.json", "r") as f:
    payload = json.load(f)

# الحفاظ على دقة معرّفات الخلايا
for neuron in payload["neurons"]:
    neuron["flywire_id"] = str(neuron["flywire_id"])

response = requests.post(
    API + "/ingest",
    headers={"Authorization": f"Bearer {token}"},
    json=payload,
    timeout=30
)

print("Status:", response.status_code)
print(response.json())

FlyMind API Token: ··········
Status: 202
{'ok': True, 'received': True, 'experiment': 'sugarR_75', 'neurons': 29, 'intervals': 20, 'totalSpikes': 23654, 'stored': False}


The `.parquet` file created during a simulation contains all spikes events of all neurons in the model.
We load the data again from disk by passing a list of result files to the `utl.load_exps` function.

We can see from the size of the dataframe
that more than 400 000 spikes were generated by activating the sugar neurons (30 trials, 1 s each).

In [ ]:
# load data from disk
df_spike = utl.load_exps([ './results/example/sugarR.parquet' ])
df_spike

The spike times can be converted to spike rates [Hz] via `utl.get_rate`, which requires the duration of each trial.
`utl.get_rate` returns `pandas.DataFrame` objects:
1. spike rate for each neuron (rows) in each experiment (column): `df_rate`
2. standard deviation of rate across trials: `df_rate_std`

For convenience, we can optionally pass the `flyid2name` dictionary to `utl.get_rate` in order to convert flywire IDs into
meaningful names.

We can see that only about 400 neurons show activity during the simulations.

In [ ]:
# calculate spike rate and standard deviation
df_rate, df_rate_std = utl.get_rate(df_spike, t_run=params['t_run'], n_run=params['n_run'], flyid2name=flyid2name)
# sort by spike rate
df_rate.sort_values('sugarR', ascending=False)

## Change stimulation frequency

We want to change the frequency of the stimulation of the sugar neurons.
To do so we modify the value for `r_poi` in the `default_params` dictionary and pass the altered dictionary to the `run_exp` function.

Note: Since physical quantities in `brian2` have to have the correct unit, we also need the `brian2.Hz` object
to define a frequency.

In [ ]:
# run with different frequency
params['r_poi'] = 100 * Hz

run_exp(exp_name='sugarR_100Hz', neu_exc=neu_sugar, params=params, **config)

We load the results via the `utl.load_exps` function and convert the spike events to rates with `utl.get_rate`

In [ ]:
ps = [
    './results/example/sugarR.parquet',
    './results/example/sugarR_100Hz.parquet',
]

df_spike = utl.load_exps(ps)
df_rate, df_rate_std = utl.get_rate(df_spike, t_run=params['t_run'], n_run=params['n_run'], flyid2name=flyid2name)
df_rate.sort_values('sugarR_100Hz', ascending=False, inplace=True)
df_rate

## Silencing neurons
We want to silence the most active neurons individually to see how that changes the activity patterns.
We do so by passing the neuron IDs we want to silence as a list `run_exp` via the `neu_slnc` argument.
In the following example, we are silencing a single neuron `[ i ]` while exciting the sugar neurons `neu_sugar`.
We can then investigate how silencing of each individual neuron affects the firing rate of a given neuron, say, MN9.

In [ ]:
#First, let's check on the MN9 firing rate when no neurons are silenced.
id_mn9 = 720575940660219265 #id for MN9
x = df_rate.loc[id_mn9, "sugarR_100Hz"]
print(f'Rate for neuron {id_mn9} is {x}')

In [ ]:
# IDs of 3 most active neurons. These neurons are all sugar-sensing neurons.
ids = df_rate.sort_values('sugarR_100Hz', ascending=False).index[:3]

for i in ids:
    run_exp(exp_name=f'sugarR-{i}', neu_exc=neu_sugar, neu_slnc=[ i ], params=params, **config)

In [ ]:
# output files
ps = [ f'./results/example/sugarR-{i}.parquet' for i in ids ]

# calculate spike rate and sort
df_spike = utl.load_exps(ps)
df_rate, df_rate_std = utl.get_rate(df_spike, t_run=params['t_run'], n_run=params['n_run'])
df_rate.loc[id_mn9, :].sort_values(ascending=True)